In [ ]:
import xarray as xr
import geopandas as gpd
from geocube.api.core import make_geocube
import numpy as np
from rasterio.plot import show
import rasterio as rio
import matplotlib.pyplot as plt
from functools import partial
from geocube.rasterize import rasterize_image
from rasterio.enums import Resampling
from shapely.geometry import Polygon
import shapely
from atlite.gis import padded_transform_and_shape

from osgeo import gdal

In [ ]:
def get_bounding_geometry(gdf):
    bounds=gdf.to_crs("EPSG:4326").bounds
    polygon = Polygon(
    [
        (bounds.minx-0.1, bounds.miny-0.1),
        (bounds.minx-0.1, bounds.maxy+0.1),
        (bounds.maxx+0.1, bounds.maxy+0.1),
        (bounds.maxx+0.1, bounds.miny-0.1),
        (bounds.minx-0.1, bounds.miny-0.1),
    ]
    )
    polygon=shapely.segmentize(polygon, max_segment_length=0.5)
    
    b=gpd.GeoDataFrame(geometry=[polygon],crs="EPSG:4326")
    
    return b.to_crs("EPSG:3035")

In [ ]:
def dem_reproject():
    dem=rio.open(snakemake.input.copernicus)
    gdf=gpd.read_file(snakemake.input.shapefile).dissolve()
    transform, shape = padded_transform_and_shape(rio.features.bounds(get_bounding_geometry(gdf)), 90)
    kwargs = dem.meta.copy()
    kwargs.update({
        'crs': "EPSG:3035",
        'transform': transform,
        'width': shape[1],
        'height': shape[0]})
    kwargs["compress"]="ZSTD"
    out_path=snakemake.output.reproj_DEM
    with rio.open(out_path, 'w', **kwargs) as dst:
                rio.warp.reproject(
                    source=rio.band(dem, 1),
                    destination=rio.band(dst, 1),
                    src_transform=dem.transform,
                    src_crs=dem.crs,
                    dst_transform=transform,
                    dst_crs="EPSG:3035",
                    resampling=Resampling.nearest)
    


In [ ]:
def dem2slope():
    dem_input = gdal.Open(snakemake.output.reproj_DEM)
    gdal.DEMProcessing(snakemake.output.temp_slope, dem_input, "slope", computeEdges=True)


In [ ]:
def wind_slope():
    max_slope=snakemake.params.maximum_slope
    slope=rio.open(snakemake.output.temp_slope)
    crs=slope.crs
    transform=slope.transform
    slope=slope.read(1)
    
    slope[slope>max_slope]=1
    slope[slope!=1]=0
    
    with rio.open(
        snakemake.output.slope_max,
        mode="w",
        driver="GTiff",
        height=slope.shape[0],
        width=slope.shape[1],
        count=1,
        dtype="uint8",
        crs=crs,
        transform=transform,
        compress='lzw',
        ) as new_dataset:
            new_dataset.write(slope.astype("uint8"), 1)
    
    


In [ ]:
dem_reproject()
dem2slope()
wind_slope()